# Environment Setup

In [ ]:
!uv pip install bitsandbytes xformers triton unsloth vllm==0.10.2
!uv pip install transformers==4.55.4
!uv pip install --no-deps trl==0.22.2

Using Python 3.12.12 environment at: /usr
Resolved 166 packages in 1.11s
Prepared 50 packages in 10.09s
Uninstalled 9 packages in 520ms
Installed 50 packages in 81ms
 + astor==0.8.1
 + bitsandbytes==0.48.1
 + blake3==1.0.8
 + cbor2==5.7.0
 - click==8.3.0
 + click==8.2.1
 + compressed-tensors==0.11.0
 + cut-cross-entropy==25.1.1
 - datasets==4.0.0
 + datasets==4.2.0
 + depyf==0.19.0
 + diskcache==5.6.3
 + dnspython==2.8.0
 + email-validator==2.3.0
 + fastapi-cli==0.0.14
 + fastapi-cloud-cli==0.3.1
 + gguf==0.17.1
 + httptools==0.7.1
 + interegular==0.3.3
 - lark==1.3.0
 + lark==1.2.2
 + llguidance==0.7.30
 - llvmlite==0.43.0
 + llvmlite==0.44.0
 + lm-format-enforcer==0.11.3
 + mistral-common==1.8.5
 + msgspec==0.19.0
 + ninja==1.13.0
 - numba==0.60.0
 + numba==0.61.2
 + openai-harmony==0.0.4
 + outlines-core==0.2.11
 + partial-json-parser==0.2.1.1.post6
 + prometheus-fastapi-instrumentator==7.1.0
 - pyarrow==18.1.0
 + pyarrow==21.0.0
 + pybase64==1.4.2
 + pycountry==24.6.1
 + pydantic-e

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [ ]:
%cd '/content/drive/MyDrive/multi-reward-math-reasoning/Llama'

/content/drive/MyDrive/multi-reward-math-reasoning/Llama


In [ ]:
%ls

 grpo_trainer_lora_model/        Llama.ipynb
 huggingface_tokenizers_cache/   unsloth_compiled_cache/
'Llama-3.2-1b outputs'/          unsloth_training_checkpoints/
 Llama-3.2-3B.ipynb              wandb/


In [ ]:
from unsloth import FastLanguageModel
from trl import SFTConfig, GRPOConfig, SFTTrainer, GRPOTrainer
from vllm import SamplingParams

import gc
import re
import time
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm.notebook import tqdm
from datasets import load_dataset, Dataset
from safetensors import safe_open

from peft import LoraConfig, get_peft_model, TaskType
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TextStreamer
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 10-21 07:47:32 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Model Setup

In [ ]:
# Try 3B first, fallback to 1B if OOM
try:
    model_id = 'unsloth/Llama-3.2-3B'          # Attempt 3B model
    model_name = model_id.split('/')[-1].lower()
    max_seq_length = 2048
    lora_rank = 8                               # Further reduced rank for memory
    print(f"Attempting to load {model_id}")
except Exception as e:
    print(f"3B model failed, falling back to 1B: {e}")
    model_id = 'unsloth/Llama-3.2-1B'          # Fallback to 1B
    model_name = model_id.split('/')[-1].lower()
    max_seq_length = 2048
    lora_rank = 16                              # Can use higher rank with 1B

Attempting to load unsloth/Llama-3.2-3B


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_id,
    max_seq_length=max_seq_length,
    load_in_4bit=True,          # Enable 4-bit quantization to save memory
    fast_inference=True,        # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.7, # Reduced from 0.9 to leave more headroom
)
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,                          # Rank: adaptation capacity (16 good for reasoning tasks)
    lora_alpha=lora_rank * 2,             # Scaling factor (typically 2x rank)
    lora_dropout=0.1,                     # Regularization to prevent overfitting
    target_modules=[                      # Reduced target modules to save memory
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        # 'gate_proj', 'up_proj', 'down_proj',  # Comment out MLP layers to save memory
    ],
    use_gradient_checkpointing='unsloth', # Reduces memory usage
    random_state=3407,
)

INFO 10-21 07:47:52 [vllm_utils.py:694] Unsloth: Patching vLLM v1 graph capture
INFO 10-21 07:47:52 [vllm_utils.py:722] Unsloth: Patching vLLM v0 graph capture
==((====))==  Unsloth 2025.10.7: Fast Llama patching. Transformers: 4.55.4. vLLM: 0.10.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.557 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: vLLM loading unsloth/llama-3.2-3b-unsloth-bnb-4bit with actual GPU utilization = 69.2%
Unsloth: Your GPU has CUDA compute capability 8.0 with VRAM = 39.56 GB.
Unsloth: Using conservativeness = 1.0. Chunked prefill tokens = 2048. Num Sequences = 320.
Unsloth: vLLM's KV Cache can use up to 24.98 GB. Also swap space = 6 GB.
WARNING 10-21 07:48:02 [c

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

...

INFO 10-21 07:50:35 [gpu_worker.py:391] Free memory on device (39.03/39.56 GiB) on startup. Desired GPU memory utilization is (0.6919826757536155, 27.37 GiB). Actual usage is 2.26 GiB for weight, 1.48 GiB for peak activation, 0.02 GiB for non-torch memory, and 1.85 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=23212828979` to fit into requested memory, or `--kv-cache-memory=35729536512` to fully utilize gpu memory. Current kv cache memory in use is 25351924019 bytes.
INFO 10-21 07:50:35 [core.py:218] init engine (profile, create kv cache, warmup model) took 110.11 seconds
INFO 10-21 07:50:37 [llm.py:295] Supported_tasks: ('generate',)
INFO 10-21 07:50:37 [__init__.py:36] No IOProcessor plugins requested by the model
Unsloth: Just some info: will skip parsing ['pre_feedforward_layernorm', 'layer_norm2', 'norm2', 'norm1', 'input_layernorm', 'post_layernorm', 'post_attention_layernorm', 'post_feedforward_layernorm', 'q_norm', 'ffn_norm', 'layer_no

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2025.10.7 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


# Chat Template

In [ ]:
# Define structured output format for mathematical reasoning
REASONING_START = '<THINK>'   # Begin reasoning section
REASONING_END = '</THINK>'    # End reasoning section
SOLUTION_START = '<SOLUTION>' # Begin final answer
SOLUTION_END = '</SOLUTION>'  # End final answer

# System prompt that teaches the model our desired reasoning structure
SYSTEM_PROMPT = f'''You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between {REASONING_START} and {REASONING_END}.
2. Provide your final numerical answer between {SOLUTION_START} and {SOLUTION_END}.
3. Be precise and show all calculation steps clearly.'''
print(SYSTEM_PROMPT)

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.


In [ ]:
chat_template = ( # Build and assign chat_template to the tokenizer
    # If the very first message is a SYSTEM role, print it + <eos>:
    "{% if messages[0]['role'] == 'system' %}"
      "{{ messages[0]['content'] + eos_token }}"
      "{% set loop_messages = messages[1:] %}"
    "{% else %}"
      # Otherwise, inject our system_prompt + <eos>:
      "{{ '{system_prompt}' + eos_token }}"
      "{% set loop_messages = messages %}"
    "{% endif %}"

    # Now loop over the remaining messages (either user or assistant):
    "{% for message in loop_messages %}"
      "{% if message['role'] == 'user' %}"
        "{{ message['content'] }}"
      "{% elif message['role'] == 'assistant' %}"
        "{{ message['content'] + eos_token }}"
      "{% endif %}"
    "{% endfor %}"

    # If we asked for "add_generation_prompt", append <REASONING> to the end:
    "{% if add_generation_prompt %}{{ '{reasoning_start}' }}"
    "{% endif %}"
)
# Replace with out specific template:
tokenizer.chat_template = chat_template\
    .replace("'{system_prompt}'",   f"'{SYSTEM_PROMPT}'")\
    .replace("'{reasoning_start}'", f"'{REASONING_START}'")

In [ ]:
example_messages = [ # Quick sanity check of the template
    {'role': 'user', 'content': 'Which country has the highest population density?'},
    {'role': 'assistant', 'content': (
        f'{REASONING_START}'
        'I know that country X is small in area but has a huge population, '
        'so its people per square kilometer is extremely high.'
        f'{REASONING_END}{SOLUTION_START}Monaco{SOLUTION_END}'
    )},
    {'role': 'user', 'content': 'Which planet is farthest from the Sun?'},
]
print(tokenizer.apply_chat_template(example_messages, tokenize=False, add_generation_prompt=True))

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Which country has the highest population density?<THINK>I know that country X is small in area but has a huge population, so its people per square kilometer is extremely high.</THINK><SOLUTION>Monaco</SOLUTION><|end_of_text|>Which planet is farthest from the Sun?<THINK>


# Pre Fine-tuning (SFT)

## Data preparation

In [ ]:
# Use a subset of NVIDIA's Open Math Reasoning dataset, which was filtered to only include high quality DeepSeek R1 traces
sft_dataset = load_dataset('unsloth/OpenMathReasoning-mini', split='cot').to_pandas()
sft_dataset = sft_dataset[['expected_answer', 'problem', 'generated_solution']]

# Try converting to number - if not, replace with NaN
is_number = pd.to_numeric(pd.Series(sft_dataset['expected_answer']), errors='coerce').notnull()
sft_dataset = sft_dataset.iloc[np.where(is_number)[0]] # Select only numbers
sft_dataset

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet:   0%|          | 0.00/106M [00:00<?, ?B/s]

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

,expected_answer,problem,generated_solution
0,14,Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$...,"<think>\nOkay, let's see. I need to solve the ..."
6,-2,Find the value of the parameter $a$ for which ...,"<think>\nOkay, so I need to find the value of ..."
9,18,What is the sum of all real numbers $x$ for wh...,"<think>\nOkay, so I need to solve the equation..."
13,2,Evaluate the sum \(\sum_{n=1}^\infty \frac{\ph...,"<think>\nOkay, so I need to evaluate the infin..."
17,30,What is the largest positive integer that divi...,"<think>\nAlright, so I need to find the larges..."
...,...,...,...
19243,244,"Let \( p \), \( q \), and \( r \) be the disti...","<think>\nOkay, so I need to find the value of ..."
19245,1,A bug is on the $0$ of a number line. At any p...,"<think>\nOkay, so I have this problem where a ..."
19247,4,A bus left point X for point Y. Two hours late...,"<think>\nOkay, let's tackle this problem step ..."
19248,18,Each interior angle of a regular n-gon measure...,"<think>\nOkay, let's see. I need to find the n..."


In [ ]:
def format_dataset(x): # Format the dataset to follow our GRPO style formatting
    expected_answer = x['expected_answer']
    problem = x['problem']

    # Remove generated <think> and </think>
    thoughts = x['generated_solution'].replace('<think>', '').replace('</think>', '')
    thoughts = thoughts.strip()

    # Add our custom formatting
    final_prompt = REASONING_START + thoughts + REASONING_END + \
                   SOLUTION_START + expected_answer + SOLUTION_END
    return [
        {'role': 'system'   , 'content': SYSTEM_PROMPT},
        {'role': 'user'     , 'content': problem},
        {'role': 'assistant', 'content': final_prompt},
    ]

sft_dataset['messages'] = sft_dataset.apply(format_dataset, axis=1)
print(tokenizer.apply_chat_template(sft_dataset['messages'][0], tokenize=False))

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Given $\sqrt{x^2+165}-\sqrt{x^2-52}=7$ and $x$ is positive, find all possible values of $x$.<THINK>Okay, let's see. I need to solve the equation √(x² + 165) - √(x² - 52) = 7, and find all positive values of x. Hmm, radicals can be tricky, but maybe if I can eliminate the square roots by squaring both sides. Let me try that.

First, let me write down the equation again to make sure I have it right:

√(x² + 165) - √(x² - 52) = 7.

Okay, so the idea is to isolate one of the radicals and then square both sides. Let me try moving the second radical to the other side:

√(x² + 165) = 7 + √(x² - 52).

Now, if I square both sides, maybe I can get rid of the square roots. Let's do that:

(√(x² + 165))² = (7 + √(x² - 52))².

S

In [ ]:
# Truncate pre fine-tuning sft_dataset to max_seq_length / 2 since we don't want too long reasoning traces
sft_dataset['seq_length'] = sft_dataset['messages'].apply(lambda x: len(tokenizer.apply_chat_template(x)))
print('Token-length percentiles (50/90/99):', np.percentile(sft_dataset['seq_length'], [50, 90, 99]))

threshold = max_seq_length / 2
sft_dataset_filtered = sft_dataset.loc[sft_dataset['seq_length'] <= threshold].copy()
print(f'Remaining for training (<= {threshold} tokens): {len(sft_dataset_filtered)}/{len(sft_dataset)}')

sft_dataset_filtered['text'] = tokenizer.apply_chat_template(sft_dataset_filtered['messages'].values.tolist(), tokenize=False)
sft_dataset_filtered = Dataset.from_pandas(sft_dataset_filtered)
sft_dataset_filtered

Token-length percentiles (50/90/99): [ 3469.   8310.4 14578.9]
Remaining for training (<= 1024.0 tokens): 73/7507


Dataset({
    features: ['expected_answer', 'problem', 'generated_solution', 'messages', 'seq_length', 'text', '__index_level_0__'],
    num_rows: 73
})

## Pre fine-tune to understand custom GRPO formatting

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=sft_dataset_filtered,
    args=SFTConfig(
        dataset_text_field='text',
        num_train_epochs=3,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        optim='adamw_8bit',
        weight_decay=0.01,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_steps=5,
        logging_steps=5,
        seed=3407,
        report_to='none', # Use this for WandB
    )
)
trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/73 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 73 | Num Epochs = 3 | Total steps = 219
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 1 x 1) = 1
 "-____-"     Trainable parameters = 4,587,520 of 3,217,337,344 (0.14% trained)


Step,Training Loss
5,1.315900
10,1.181600
15,1.145200
20,1.022000
25,0.924100
30,0.892700
35,0.903300
40,0.795700
45,0.779000
50,0.781000


Unsloth: Will smartly offload gradients to save VRAM!


TrainOutput(global_step=219, training_loss=0.7088047195242965, metrics={'train_runtime': 114.3662, 'train_samples_per_second': 1.915, 'train_steps_per_second': 1.915, 'total_flos': 3348430032273408.0, 'train_loss': 0.7088047195242965, 'epoch': 3.0})

## Check if model has learnt to follow the format

In [ ]:
text = tokenizer.apply_chat_template( # Render into a single string and append <REASONING> for generation
    sft_dataset_filtered[1]['messages'][:2],
    tokenize=False, add_generation_prompt=True, # Append the final <REASONING>
)
_ = model.generate(
    **tokenizer(text, return_tensors='pt').to('cuda'),
    temperature=0, max_new_tokens=1024,
    streamer=TextStreamer(tokenizer, skip_prompt=False), # Stream the model's generations (CoT + solution)
)

<|begin_of_text|>You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>What is the average book width, in centimeters, of five books with the following widths: $6$, $\frac{1}{2}$, $1$, $2.5$, and $10$?<THINK>Okay, let's see. I need to find the average width of five books with widths 6, 1/2, 1, 2.5, and 10 centimeters. Hmm, average. Let me think. Average is the sum of all the values divided by the number of values. So, the sum of these five widths would be 6 + 1/2 + 1 + 2.5 + 10. Let me add them up.

6 + 1/2 is 6.5. Then 6.5 + 1 is 7.5. Then 7.5 + 2.5 is 10. Then 10 + 10 is 20. So the sum is 20 centimeters.

Now, the average is 20 divided by 5, which is 20 divided by 5. Let me calculate that. 20 divided by 5 is 4. So the average width is 4 centimeters. Wait, but let me 

In [ ]:
del sft_dataset, sft_dataset_filtered
gc.collect()
torch.cuda.empty_cache()

# Post Fine-tuning (RL)

## Data preparation

In [ ]:
def process_dataset_sample(example): # Convert GSM8K example to conversation format for GRPO training
    return {
        'prompt': [ # Create conversation with system prompt for structured reasoning
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': example['question']},
        ],
        # Extract numerical answer from GSM8K format ('Explanation... #### 42') as Ground truth for reward functions
        'answer': example['answer'].split('####')[1].strip() if '####' in example['answer'] else None
    }

In [ ]:
# train_dataset = load_dataset('openai/gsm8k', 'main', split=['train[:10%]'])
train_dataset = load_dataset('openai/gsm8k', 'main', split='train')
train_dataset = train_dataset.map(process_dataset_sample)

print(f'Training samples: {len(train_dataset):,}\n'
      f"- Sample question: {train_dataset[0]['prompt'][1]['content']}\n"
      f"- Sample answer (ground truth for rewards): {train_dataset[0]['answer']}\n"
      f"- Prompt (system + user):\n{train_dataset[0]['prompt']}")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Training samples: 7,473
- Sample question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
- Sample answer (ground truth for rewards): 72
- Prompt (system + user):
[{'content': 'You are a mathematical reasoning assistant. When given a math problem:\n1. Show your step-by-step work between <THINK> and </THINK>.\n2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.\n3. Be precise and show all calculation steps clearly.', 'role': 'system'}, {'content': 'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?', 'role': 'user'}]


In [ ]:
# Get the top 90% prompt length so we don't accidentally truncate them, i.e. we'll remove the top 10% long prompts
tokenized_dataset = train_dataset.map(
    lambda x: {'tokens': tokenizer.apply_chat_template(x['prompt'], add_generation_prompt=True, tokenize=True)},
    batched=True,
).map(lambda x: {'length': len(x['tokens'])})
print(tokenizer.decode(tokenized_dataset[0]['tokens']))

thresholds = np.percentile(tokenized_dataset['length'], [50, 90, 99])
max_prompt_length = int(thresholds[1])
print('Token-length percentiles (50/90/99):', thresholds, '=> Choose max_prompt_length =', max_prompt_length)

# Filter only samples smaller than 90% max length
train_dataset = train_dataset.select(np.where(np.array(tokenized_dataset['length']) <= max_prompt_length)[0])
print(f'Remaining for training (<= {max_prompt_length} tokens): {len(train_dataset)}/{len(tokenized_dataset)}')
del tokenized_dataset

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

You are a mathematical reasoning assistant. When given a math problem:
1. Show your step-by-step work between <THINK> and </THINK>.
2. Provide your final numerical answer between <SOLUTION> and </SOLUTION>.
3. Be precise and show all calculation steps clearly.<|end_of_text|>Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<THINK>
Token-length percentiles (50/90/99): [116. 149. 187.] => Choose max_prompt_length = 149
Remaining for training (<= 149 tokens): 6749/7473


## Regex Patterns

In [ ]:
# Match the reasoning sections and answers
match_format = re.compile(
    # rf'^[\s]{{0,}}'                                     # Optional whitespace at start
    # rf'{REASONING_START}.+?{REASONING_END}.*?'          # Reasoning section (non-greedy)
    rf'{REASONING_END}.*?'                              # We always prepend REASONING_START
    rf'{SOLUTION_START}(.+?){SOLUTION_END}'             # Solution section with capture group
    rf'[\s]{{0,}}(?:{re.escape(tokenizer.eos_token)})?' # Add optional EOS token matching
    rf'[\s]{{0,}}$',                                    # Optional whitespace at end
    flags=re.MULTILINE | re.DOTALL,                     # Multi-line matching with . matching newlines
)
match_format.findall( # Verify it works
    f'{REASONING_START}Let me think!{REASONING_END}'\
    f'{SOLUTION_START}\n2\n{SOLUTION_END}\n\n',
)

['\n2\n']

In [ ]:
# Sometimes it might not be 1 number as the answer, but like a sentence.
# For example: 'The solution is $20' -> we extract 20
# We also remove possible commas for example as in 123,456
match_number = re.compile(
    rf'{SOLUTION_START}.*?[\s]{{0,}}([-]?[\d\.\,]{{1,}})', # Extract numbers from solution section
    flags=re.MULTILINE | re.DOTALL | re.IGNORECASE,        # Flexible pattern matching
)
print(match_number.findall('<SOLUTION>  0.34  </SOLUTION>'))
print(match_number.findall('<SOLUTION>  123,456  </SOLUTION>'))
print(match_number.findall('<SOLUTION>  -0.234  </SOLUTION>'))
print(match_number.findall('<SOLUTION>17</SOLUTION>'))

['0.34']
['123,456']
['-0.234']
['17']


## Multi-reward design

In [ ]:
def match_format_strictly(completions, **kwargs) -> list[float]:
    ''' Reward Function 1: Exact Format Compliance
    High reward (3.0) for perfect format adherence
    Ensures model learns the complete structured output pattern
    '''
    return [
        3.0 if match_format.search(completion[0]['content']) else 0.0
        for completion in completions
    ]

In [ ]:
# If it fails, reward the model if it at least follows the format partially, by counting each symbol
def match_format_softly(completions, **kwargs) -> list[float]:
    ''' Reward Function 2: Partial Format Credit
    Graduated scoring for format elements
    Encourages learning individual components even if not perfect
    '''
    rewards = []
    for completion in completions:
        reward = 0
        response = completion[0]['content']

        # Count how many keywords are seen - we penalize if too many!
        # Award +0.5 for correct token count, -0.5 for wrong count
        # No need to reward REASONING_START since we always prepend it!
        # reward += 0.5 if response.count(REASONING_START) == 1 else -0.5
        reward += 0.5 if response.count(REASONING_END) == 1 else -0.5
        reward += 0.5 if response.count(SOLUTION_START) == 1 else -0.5
        reward += 0.5 if response.count(SOLUTION_END) == 1 else -0.5
        rewards.append(reward)
    return rewards

In [ ]:
# Extract the generated answer, and reward or penalize it
def check_answer_correctness(completions, answer, **kwargs) -> list[float]:
    ''' Reward Function 3: Graduated scoring for mathematical accuracy
    - 5.0: Exact string match gets full points
    - 2.0: Within 10% (close answer)
    - 1.5: Within 20% (reasonable attempt)
    - -2.5: Wrong answer (penalty for incorrect math)
    '''
    responses = [completion[0]['content'] for completion in completions]
    extracted_responses = [ # Extract answers using format pattern
        guess.group(1) if (guess := match_format.search(r)) else None
        for r in responses
    ]
    rewards = []
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable answer
            rewards.append(-2.0)
            continue

        if guess == true_answer: rewards.append(5.0)                   # Correct answer gets 5 points!
        elif guess.strip() == true_answer.strip(): rewards.append(3.5) # Match if spaces are seen, but less reward
        else: # Try numerical comparison for partial credit
            try: # We also reward it based on how close the answer is to the true one via ratios
                ratio = float(guess) / float(true_answer)     # If the answer is within some range, reward it!
                if 0.9 <= ratio <= 1.1: rewards.append(2.0)   # Within 10%
                elif 0.8 <= ratio <= 1.2: rewards.append(1.5) # Within 20%
                else: rewards.append(-2.5)                    # Penalize wrong answers
            except (ValueError, ZeroDivisionError):
                rewards.append(-4.5)                          # Invalid numerical format
    return rewards

In [ ]:
def check_numbers_extraction(prompts, completions, answer, **kwargs) -> list[float]:
    ''' Reward Function 4: Number Extraction Ability
    Tests the model's ability to extract numerical values from solution sections
    Complementary to exact format matching - focuses on parsing capability
    '''
    question = prompts[0][-1]['content'] # Exclude system prompt
    responses = [completion[0]['content'] for completion in completions]

    extracted_responses = [ # Extract numbers from solution sections using number pattern
        guess.group(1) if (guess := match_number.search(r)) else None
        for r in responses
    ]
    rewards = []

    # Print only every few steps
    check_numbers_extraction.counter = getattr(check_numbers_extraction, 'counter', 0) + 1
    if check_numbers_extraction.counter % 100 == 0:
        print(
            '==' * 100,
            f'\nQuestion: {question}'
            f'\nPrediction: {extracted_responses[0]}, GT Answer: {answer[0]}'
            f'\nResponse:\n{responses[0]}'
        )
    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None: # No extractable number
            rewards.append(-2.5)
            continue

        try: # Simple numerical equality check
            true_val = float(true_answer.strip())             # Convert to numbers
            guess_val = float(guess.strip().replace(',', '')) # Remove commas like in 123,456
            rewards.append(3.5 if guess_val == true_val else -1.5)
        except (ValueError, TypeError):
            rewards.append(0) # Invalid number format
    return rewards

## GRPO training setup

In [ ]:
max_prompt_length = 152 + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

In [ ]:
training_args = GRPOConfig(          # Configure GRPO training parameters for mathematical reasoning
    output_dir=f'/tmp/{model_name}', # Directory for checkpoints and logs
    vllm_sampling_params=vllm_sampling_params,
    # Training speed control
    num_train_epochs=1,              # Total number of training epochs
    per_device_train_batch_size=1,   # Keep at 1 for 3B model
    gradient_accumulation_steps=8,   # Reduced from 16 to 8 for memory
    # Computing the loss: https://huggingface.co/docs/trl/main/grpo_trainer#computing-the-loss
    scale_rewards='batch',           # Calculate mean at local/group level and std at global/batch level enables more robust reward shaping
    loss_type='dr_grpo',             # Fully remove response length bias, dividing by a constant instead of the sequence length
    # Precision & Optimization
    optim='adamw_8bit',              # adamw_torch_fused, adamw_8bit, paged_adamw_8bit
    weight_decay=0.1,                # Regularization
    max_grad_norm=0.1,               # Aggressive gradient clipping for stable training
    gradient_checkpointing=True,
    bf16=torch.cuda.is_available(),  # Enable mixed-precision training if a CUDA GPU is available (faster, less memory)
    # Learning rate scheduling
    learning_rate=1e-5,              # Conservative LR to prevent destabilizing reasoning
    warmup_ratio=0.1,
    lr_scheduler_type='cosine_with_min_lr',
    lr_scheduler_kwargs=dict(min_lr=1e-6),
    # Generation control - REDUCED FOR MEMORY
    temperature=1.0,
    num_generations=2,                           # Reduced from 2 to 1 to save memory
    max_prompt_length=max_prompt_length,         # Default: 512. Sufficient for complex word problems
    max_completion_length=max_completion_length, # Default: 256. Room for detailed step-by-step reasoning
    # Reporting and saving
    report_to='wandb',
    logging_steps=10,
    logging_strategy='steps',
    save_total_limit=1,
    # max_steps=100,                               # TEMPORARY: Limit steps for testing
    # For optional evaluation
    # per_device_eval_batch_size=4,
    # bf16_full_eval=torch.cuda.is_available(),
    # eval_strategy='steps',                       # Evaluate after each epoch
    # load_best_model_at_end=True,                 # Load the best model based on validation loss
)

Unsloth: We now expect `per_device_train_batch_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 2


## Train the model

In [ ]:
# Memory management and cleanup before training
import gc
import torch

# Clear any existing cache
torch.cuda.empty_cache()
gc.collect()

# Set CUDA memory allocation configuration for better memory management
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Check current memory usage
print(f"GPU Memory before training:")
print(f"  Allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"  Reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")
print(f"  Free: {torch.cuda.get_device_properties(0).total_memory/1024**3 - torch.cuda.memory_reserved()/1024**3:.2f} GB")

GPU Memory before training:
  Allocated: 26.12 GB
  Reserved: 26.37 GB
  Free: 13.19 GB


In [ ]:
%%time
trainer = GRPOTrainer(            # Initialize GRPO trainer with multi-reward system
    model=model,                  # LoRA-adapted quantized model
    processing_class=tokenizer,
    train_dataset=train_dataset,  # Processed GSM8K dataset
    args=training_args,           # Training configuration
    reward_funcs=[                # 4 complementary reward functions
        match_format_strictly,    # Perfect structure compliance
        match_format_softly,      # Partial format credit
        check_answer_correctness, # Mathematical accuracy
        check_numbers_extraction, # Number parsing ability
    ]
)
trainer.train()
trainer.save_model(f'./{model_name}_grpo')

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 6,749 | Num Epochs = 1 | Total steps = 843
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 4,587,520 of 3,217,337,344 (0.14% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: walteryeyint (walteryeyint-university-of-technology-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_strictly / mean,rewards / match_format_strictly / std,rewards / match_format_softly / mean,rewards / match_format_softly / std,rewards / check_answer_correctness / mean,rewards / check_answer_correctness / std,rewards / check_numbers_extraction / mean,rewards / check_numbers_extraction / std
10,0.360700,0.346875,5.659392,1365.043750,654.500000,1895.000000,0.293750,1148.610498,654.500000,1785.100000,0.374448,2.062500,1.405407,0.618750,1.354887,-1.212500,2.380370,-1.121875,1.791664
20,0.321100,0.356250,5.847365,1358.075000,660.200000,1895.000000,0.293750,1145.573334,660.200000,1671.600000,0.354318,1.968750,1.399453,0.506250,1.377157,-1.025000,2.522994,-1.093750,1.818494
30,0.327000,0.990625,5.914369,1351.550000,595.400000,1895.000000,0.256250,1173.176257,595.400000,1738.800000,0.462789,2.043750,1.395383,0.556250,1.366210,-0.615625,2.489847,-0.993750,1.758234
40,0.470800,0.078125,5.718722,1379.143750,666.700000,1895.000000,0.293750,1165.799243,666.700000,1702.400000,0.356088,2.025000,1.432876,0.518750,1.437528,-1.228125,2.419820,-1.237500,1.760871
50,0.228900,1.165625,6.003818,1281.693750,621.100000,1886.400000,0.218750,1106.038629,621.100000,1661.800000,0.597597,2.156250,1.323523,0.718750,1.239616,-0.821875,2.712603,-0.887500,1.931800
60,0.476600,0.915625,5.235532,1250.293750,586.800000,1895.000000,0.200000,1089.582721,586.800000,1727.900000,0.376789,2.250000,1.318222,0.825000,1.223294,-1.118750,2.310123,-1.040625,1.596127
70,0.319600,2.712500,5.711444,1142.925000,623.300000,1809.500000,0.106250,1064.039746,623.300000,1646.100000,0.374796,2.643750,0.798553,1.156250,0.777286,-0.450000,3.093379,-0.637500,2.065011
80,0.309100,2.153125,6.098259,1109.031250,559.000000,1846.000000,0.162500,953.492389,559.000000,1517.300000,0.389397,2.325000,1.192223,0.831250,1.142153,-0.456250,2.860729,-0.546875,2.049269
90,0.322100,2.159375,5.904838,1112.775000,505.900000,1890.200000,0.143750,978.775977,505.900000,1623.700000,0.384641,2.437500,1.170383,0.993750,1.058712,-0.534375,2.982624,-0.737500,2.033955
100,0.261800,1.743750,5.646428,1041.475000,491.500000,1895.000000,0.125000,921.819800,491.500000,1566.200000,0.437234,2.475000,1.134908,0.968750,1.101495,-0.881250,2.792310,-0.818750,1.928621


Question: The tallest building in the world is 100 feet tall.  If the second tallest is half that tall, and the third tallest is half as tall as the second, and the fourth is one-fifth as tall as the third, how tall are all 4 buildings put together?
Prediction: 185, GT Answer: 180
Response:
Okay, let's see. The problem said that the tallest building is 100 feet tall. The next tallest one is half that, then the next one is half as tall as the second, and the fourth one is one-fifth as tall as the third. Hmm, I need to find the total height of all four buildings. So first, let's start by breaking this down step by step.

First, the tallest building is 100 feet tall. That's building #1. The second tallest building is half that, which is 50 feet. Building #2. The third tallest is half as tall as building 2, so 25 feet. Building #3. Then the fourth tallest is one-fifth as tall as building 3, which is 10 feet tall. Building #4.

So now I need to add up the heights of all these buildings. Sta

Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,kl,rewards / match_format_strictly / mean,rewards / match_format_strictly / std,rewards / match_format_softly / mean,rewards / match_format_softly / std,rewards / check_answer_correctness / mean,rewards / check_answer_correctness / std,rewards / check_numbers_extraction / mean,rewards / check_numbers_extraction / std
10,0.360700,0.346875,5.659392,1365.043750,654.500000,1895.000000,0.293750,1148.610498,654.500000,1785.100000,0.374448,2.062500,1.405407,0.618750,1.354887,-1.212500,2.380370,-1.121875,1.791664
20,0.321100,0.356250,5.847365,1358.075000,660.200000,1895.000000,0.293750,1145.573334,660.200000,1671.600000,0.354318,1.968750,1.399453,0.506250,1.377157,-1.025000,2.522994,-1.093750,1.818494
30,0.327000,0.990625,5.914369,1351.550000,595.400000,1895.000000,0.256250,1173.176257,595.400000,1738.800000,0.462789,2.043750,1.395383,0.556250,1.366210,-0.615625,2.489847,-0.993750,1.758234
40,0.470800,0.078125,5.718722,1379.143750,666.700000,1895.000000,0.293750,1165.799243,666.700000,1702.400000,0.356088,2.025000,1.432876,0.518750,1.437528,-1.228125,2.419820,-1.237500,1.760871
50,0.228900,1.165625,6.003818,1281.693750,621.100000,1886.400000,0.218750,1106.038629,621.100000,1661.800000,0.597597,2.156250,1.323523,0.718750,1.239616,-0.821875,2.712603,-0.887500,1.931800
60,0.476600,0.915625,5.235532,1250.293750,586.800000,1895.000000,0.200000,1089.582721,586.800000,1727.900000,0.376789,2.250000,1.318222,0.825000,1.223294,-1.118750,2.310123,-1.040625,1.596127
70,0.319600,2.712500,5.711444,1142.925000,623.300000,1809.500000,0.106250,1064.039746,623.300000,1646.100000,0.374796,2.643750,0.798553,1.156250,0.777286,-0.450000,3.093379,-0.637500,2.065011
80,0.309100,2.153125,6.098259,1109.031250,559.000000,1846.000000,0.162500,953.492389,559.000000,1517.300000,0.389397,2.325000,1.192223,0.831250,1.142153,-0.456250,2.860729,-0.546875,2.049269
90,0.322100,2.159375,5.904838,1112.775000,505.900000,1890.200000,0.143750,978.775977,505.900000,1623.700000,0.384641,2.437500,1.170383,0.993750,1.058712,-0.534375,2.982624,-0.737500,2.033955
100,0.261800,1.743750,5.646428,1041.475000,491.500000,1895.000000,0.125000,921.819800,491.500000,1566.200000,0.437234,2.475000,1.134908,0.968750,1.101495,-0.881250,2.792310,-0.818750,1.928621


Question: Simone ate 1/2 of an apple each day for 16 days. Lauri ate 1/3 of an apple each day for 15 days. How many apples did the two girls eat altogether?
Prediction: 13, GT Answer: 13
Response:
Okay, so I need to figure out how much food Laurie ate and how much Simone ate, and then add them together to find the total. First, Laurie ate 1/3 of an apple each day for 15 days. That means she would have eaten 15 days multiplied by 1/3 of an apple per day, which is 5 apples in total. Now, let's look at Simone. She ate 1/2 of an apple each day for 16 days. In 16 days, she would eat 16 days multiplied by 1/2 of an apple per day, so 8 apples. So Laurie ate 5 apples and Simone ate 8 apples.

To find the total, Laura ate 5 apples and Simone ate 8 apples, so the total would be 5 plus 8, which is 13 apples. Wait, let me check that again. Laurie started with 5 and added 8 more, so total is 5 + 8 = 13. Yeah, that seems right. 

So Laurie ate 5 apples and Simone ate 8 apples, totaling 13 apples. Le

# Evaluation

## Resource usage

In [ ]:
# Memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f'GPU = {gpu_stats.name}. Max memory = {max_memory} GB.')
print(f'{start_gpu_memory} GB of memory reserved.')

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.557 GB.
37.246 GB of memory reserved.


In [ ]:
# Extract runtime info
last_log = trainer.state.log_history[-1] # Final memory and time stats
train_seconds = last_log['train_runtime']
samples_per_second = last_log.get('train_samples_per_second', None)

# Recompute GPU memory stats
used_memory   = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
used_for_lora = round(used_memory - start_gpu_memory, 2)
used_pct      = round(used_memory / max_memory * 100, 2)
lora_pct      = round(used_for_lora / max_memory * 100, 2)

print(f'Training time: {train_seconds:.1f} seconds ({train_seconds / 60:.2f} minutes)')
if samples_per_second: print(f'Throughput: {samples_per_second:.1f} samples/second')
print(f'Peak VRAM usage: {used_memory} GB ({used_pct}% of max memory)')
print(f'VRAM for training: {used_for_lora} GB ({lora_pct}% of max memory)')

Training time: 25765.2 seconds (429.42 minutes)
Throughput: 0.3 samples/second
Peak VRAM usage: 37.25 GB (94.17% of max memory)
VRAM for training: 0.0 GB (0.0% of max memory)


## Verify LoRA is actually trained

In [ ]:
example_text = 'What is the sqrt of 101?'
# example_text = 'Solve (x + 2)^2 = 0'
# example_text = "How many r's are in strawberry?"

sampling_params = SamplingParams(
    temperature=1.0,
    top_k=50,
    max_tokens=max_completion_length,
)
print(model.fast_generate( # Try the model without any GRPO trained
    example_text, sampling_params=sampling_params,
    lora_request=None
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 Well, it can be found in the list of perfect squares in the table below. If 101 is a perfect square, the answer is YES, and if it is not a perfect square, the answer is NO. 100 is a perfect square because its square root is equal to 10 . The first 100 odd numbers are in the table below, and so is the square root of each number.
The square root of a number has two parts: the whole number and the decimal part. The whole part is the number of whole squares inside the square root of that number, and it is the largest whole number that can be multiplied by itself to equal the original number. The decimal part is the remaining portion after the whole part, which is equal to the square root of that number with no whole part. Note that the decimal part may consist of more than one digit. For example, the whole part of the square root of 3 is 1 and the decimal part is 0.58. Here are the first 100 odd numbers with their square roots.
What can the table below be used for?
The numbers in the odd 

In [ ]:
tensors = {}
with safe_open(f'./{model_name}_grpo/adapter_model.safetensors', framework='pt') as f:
    for key in f.keys(): # Verify both A and B are non zero
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

In [ ]:
# Debug: Check if training actually happened
print("=== Training Verification ===")
print(f"Model name: {model_name}")
print(f"Expected LoRA path: ./{model_name}_grpo/")

# Check if the adapter file exists and has reasonable size
import os
adapter_path = f'./{model_name}_grpo/adapter_model.safetensors'
if os.path.exists(adapter_path):
    file_size = os.path.getsize(adapter_path) / (1024 * 1024)  # MB
    print(f"✅ LoRA adapter found: {file_size:.1f} MB")

    # Check if we have reasonable training logs
    if hasattr(trainer, 'state') and trainer.state.log_history:
        final_loss = trainer.state.log_history[-1].get('train_loss', 'N/A')
        total_steps = trainer.state.log_history[-1].get('step', 'N/A')
        print(f"Final training loss: {final_loss}")
        print(f"Total training steps: {total_steps}")
    else:
        print("⚠️ No training logs found - model may not be properly trained")
else:
    print(f"❌ LoRA adapter not found at {adapter_path}")
    print("Available files:")
    !ls -la

=== Training Verification ===
Model name: llama-3.2-3b
Expected LoRA path: ./llama-3.2-3b_grpo/
✅ LoRA adapter found: 17.5 MB
Final training loss: 0.1506692246685284
Total training steps: 843


In [ ]:
# Debug: Test reward functions with sample outputs
print("=== Reward Function Testing ===")

# Test with a properly formatted response
good_response = f"{REASONING_START}Let me solve this step by step. 2 + 3 = 5{REASONING_END}{SOLUTION_START}5{SOLUTION_END}"
bad_response = "The answer is probably 5 or something like that."

test_completions_good = [[{'content': good_response}]]
test_completions_bad = [[{'content': bad_response}]]
test_answer = ['5']

print("Good response:", good_response)
print("Format strict reward (good):", match_format_strictly(test_completions_good))
print("Format strict reward (bad):", match_format_strictly(test_completions_bad))

print("\nSoft format rewards (good):", match_format_softly(test_completions_good))
print("Soft format rewards (bad):", match_format_softly(test_completions_bad))

print("\nAnswer correctness (good):", check_answer_correctness(test_completions_good, test_answer))
print("Answer correctness (bad):", check_answer_correctness(test_completions_bad, test_answer))

# Test regex patterns
print(f"\nRegex test - format match: {bool(match_format.search(good_response))}")
print(f"Regex test - number match: {match_number.findall(good_response)}")

=== Reward Function Testing ===
Good response: <THINK>Let me solve this step by step. 2 + 3 = 5</THINK><SOLUTION>5</SOLUTION>
Format strict reward (good): [3.0]
Format strict reward (bad): [0.0]

Soft format rewards (good): [1.5]
Soft format rewards (bad): [-1.5]

Answer correctness (good): [5.0]
Answer correctness (bad): [-2.0]

Regex test - format match: True
Regex test - number match: ['5']


In [ ]:
# Load the LoRA and test without using system prompt
# which should not (or minimal) affect the model's original reasoning ability
text = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': example_text}],
    add_generation_prompt=True, tokenize=False,
)
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Okay, so I need to determine the value of √101. Let me recall, the square root of a number x is the number that when multiplied by itself (or squared) equals x. So, if I have 101, I need to find a number whose square is 101.

Since 101 is 10 × 10 + 1, the square of 10 is 100, and 1 adds 1 more. So, is 10 the square root? No, because (10)² = 100, which is less than 101. So the next square number, 11, its square is 121. Subtract 101 from 121: 121 - 101 = 20. That means 20 is the difference between the square and the number.

Wait, the square root of 101 is 10, because 10² = 100. Then the difference between 101 and 100 is 1, so the number to subtract is 10. If I add that 10 to 10, I get 20, which is 101. So 20 is the square root.

Wait, but I'm not entirely sure if that's correct. Let me double-check: if 10² = 100, and then 101 = 100 + 1, then (20)² = (10 + 10)² = 100 + 100, which is 200. If we subtract 101 from 200, we get 199. Which is smaller than 101, so 20 is not the correct answer. 

In [ ]:
# Test using system prompt
text = tokenizer.apply_chat_template([
    {'role': 'system', 'content': SYSTEM_PROMPT},
    {'role': 'user'  , 'content': example_text},
], add_generation_prompt=True, tokenize=False)

# Compare results with system prompt but without LoRA
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=None,
)[0].outputs[0].text)

# Reasoning model is much better - it's not always correct, since we only trained it for an hour
# It'll be better if we extend the sequence length and train for longer
print(model.fast_generate(
    text, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)[0].outputs[0].text)

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

 1*1*1 is 1
1 root 101 equals 10< SOLUTION>=√10.


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Okay, so I need to find the square root of 101. Let me recall... the square root of a number x is the number whose square is x. For example, the square root of 9 is 3, because 3 squared (3 × 3) is still 9. Got it.

So, 101 is a square number. But how do I get the square root of it?

If the number is square, the square root is the number itself. Right? Because sqrt of 25 is 5, and 5 squared is indeed equal to 25. So for 101, the square root would be 10. Hmm, but maybe the problem wants me to calculate the root of 101 as decimal number.

Wait, square root of a square number is the number itself. So the square root of 101 is 10. Sure.

But just to make sure, I plug 10 into 101 and check: 10 × 10 equals 100, which is less than 101. So yes, the answer should be 10.
To find the square root of 101, we start by recalling the definition of the square root of a number x:

\[
\sqrt{x} = y \text{ if and only if } y^2 = x.
\]

Since 101 is a square number (it is equal to 101), its square root is si

## Performance on Test set

In [45]:
# test_dataset = load_dataset('openai/gsm8k', 'main', split=['test[:10%]'])
test_dataset = load_dataset('openai/gsm8k', 'main', split='test').map(process_dataset_sample)
test_texts = [
    tokenizer.apply_chat_template([
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': sample['prompt'][1]['content']},
    ], add_generation_prompt=True, tokenize=False)
for sample in test_dataset]
print(f'Testing samples:', len(test_dataset))

Testing samples: 1319


In [46]:
outputs_with_lora = model.fast_generate(
    test_texts, sampling_params=sampling_params,
    lora_request=model.load_lora(f'./{model_name}_grpo'),
)
outputs_without_lora = model.fast_generate(
    test_texts, sampling_params=sampling_params,
    lora_request=None,
)

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

In [47]:
# Compare the correct amount of using and not using LoRA
no_lora_correct_format_cnt = lora_correct_format_cnt = 0
no_lora_correct_answer_cnt = lora_correct_answer_cnt = 0
no_lora_correct_all_cnt = lora_correct_all_cnt = 0
num_test_samples = len(test_dataset)

for output_with_lora, output_without_lora, answer in zip(outputs_with_lora, outputs_without_lora, test_dataset['answer']):
    correct_format = match_format.search(output_with_lora.outputs[0].text)
    correct_answer = (guess := match_number.search(output_with_lora.outputs[0].text)) and guess.group(1) == answer
    correct_all = correct_format and correct_answer
    if correct_format: lora_correct_format_cnt += 1
    if correct_answer: lora_correct_answer_cnt += 1
    if correct_all: lora_correct_all_cnt += 1

    correct_format = match_format.search(output_without_lora.outputs[0].text)
    correct_answer = (guess := match_number.search(output_without_lora.outputs[0].text)) and guess.group(1) == answer
    correct_all = correct_format and correct_answer
    if correct_format: no_lora_correct_format_cnt += 1
    if correct_answer: no_lora_correct_answer_cnt += 1
    if correct_all: no_lora_correct_all_cnt += 1

pd.DataFrame({
    'Without LoRA': {
        'Correct Format': f'{no_lora_correct_format_cnt}/{num_test_samples} ({no_lora_correct_format_cnt / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'{no_lora_correct_answer_cnt}/{num_test_samples} ({no_lora_correct_answer_cnt / num_test_samples * 100:.2f}%)',
        'Correct Both': f'{no_lora_correct_all_cnt}/{num_test_samples} ({no_lora_correct_all_cnt / num_test_samples * 100:.2f}%)',
    },
    'With LoRA': {
        'Correct Format': f'{lora_correct_format_cnt}/{num_test_samples} ({lora_correct_format_cnt / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'{lora_correct_answer_cnt}/{num_test_samples} ({lora_correct_answer_cnt / num_test_samples * 100:.2f}%)',
        'Correct Both': f'{lora_correct_all_cnt}/{num_test_samples} ({lora_correct_all_cnt / num_test_samples * 100:.2f}%)',
    },
    'Improvement': {
        'Correct Format': f'+{lora_correct_format_cnt - no_lora_correct_format_cnt} ({(lora_correct_format_cnt - no_lora_correct_format_cnt) / num_test_samples * 100:.2f}%)',
        'Correct Answer': f'+{lora_correct_answer_cnt - no_lora_correct_answer_cnt} ({(lora_correct_answer_cnt - no_lora_correct_answer_cnt) / num_test_samples * 100:.2f}%)',
        'Correct Both': f'+{lora_correct_all_cnt - no_lora_correct_all_cnt} ({(lora_correct_all_cnt - no_lora_correct_all_cnt) / num_test_samples * 100:.2f}%)',
    }
}).T

,Correct Format,Correct Answer,Correct Both
Without LoRA,55/1319 (4.17%),11/1319 (0.83%),0/1319 (0.00%)
With LoRA,1285/1319 (97.42%),334/1319 (25.32%),328/1319 (24.87%)
Improvement,+1230 (93.25%),+323 (24.49%),+328 (24.87%)


# Inference

In [48]:
def generate_with_reasoning(questions, max_completion_length=512):
    conversations = [[                        # Format input using conversation template
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': question},
    ] for question in questions]

    prompts = [tokenizer.apply_chat_template( # Apply chat template and tokenize
        conversation,
        add_generation_prompt=True,         # Add assistant prompt
        tokenize=False,                     # Return string, not tokens
    ) for conversation in conversations]

    # Generate response with reasoning-optimized parameters
    inputs = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
    start_time = time.time()
    with torch.no_grad():
        output_ids = model.generate(           # Generate response with reasoning-optimized parameters
            **inputs,
            max_new_tokens=max_completion_length,
            temperature=0.7,                # Balance creativity and consistency
            top_p=0.9,                      # Nucleus sampling for quality
            do_sample=True,                 # Enable sampling for varied reasoning paths
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,         # Reduce repetitive reasoning steps
            length_penalty=1.0,             # Neutral preference for response length
            early_stopping=True,            # Stop at natural completion
            streamer=TextStreamer(tokenizer, skip_prompt=True),
        )
    end_time = time.time()
    inference_duration = end_time - start_time
    num_generated_tokens = output_ids.shape[1] - inputs['input_ids'].shape[1]

    output_ids = output_ids[:, inputs['input_ids'][0].shape[-1]:output_ids.shape[-1]]
    responses = tokenizer.batch_decode(output_ids, skip_special_tokens=True) # Decode and extract only the generated portion
    return responses, inference_duration, num_generated_tokens

In [49]:
test_dataset = load_dataset('openai/gsm8k', 'main', split='test').map(process_dataset_sample)
gsm8k_question = test_dataset[0]['question']
expected_answer = test_dataset[0]['answer']

print('Question:', gsm8k_question, '\nResponse:')
gsm8k_responses, inference_duration, num_generated_tokens = generate_with_reasoning([gsm8k_question], max_completion_length)
gsm8k_response = gsm8k_responses[0]
print('Inference time (secs):', inference_duration)
print('Generated tokens:', num_generated_tokens)

Question: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market? 
Response:
Okay, let's try to figure out how much money Janet makes from selling extra duck eggs at the farmers' market each day.

First, we need to know how many eggs she lays per day, which is 16. Then, since she eats 3 eggs for breakfast every morning, that leaves 13 eggs available for other uses. She bakes muffins for her friends using 4 eggs each day, so that leaves 9 eggs left over. The remaining eggs are sold at the market for $2 per egg.

To calculate the total amount of money made from selling these eggs, we would add up the revenue from each source:

Total eggs laid per day: 16
Eggs eaten for breakfast: 3
Eggs used for baking: 4
Eggs sold at market: (16 - (3 + 4)) = 9

Revenue from eggs sol

In [50]:
# Validate format compliance
has_solution = SOLUTION_START in gsm8k_response and SOLUTION_END in gsm8k_response
print('Reasoning section:', REASONING_END in gsm8k_response)
print('Solution section:', has_solution)

if has_solution: # Check answer accuracy if solution section exists
    try:
        solution_text = gsm8k_response.split(SOLUTION_START)[1].split(SOLUTION_END)[0].strip()
        extracted_number = ''.join(filter(str.isdigit, solution_text))
        expected_number = ''.join(filter(str.isdigit, expected_answer))
        print('Extracted:', solution_text)
        print('Expected:', expected_answer)
        print('Correct:', extracted_number == expected_number)
    except:
        print('Could not extract solution')

Reasoning section: True
Solution section: True
Extracted: 18
Expected: 18
Correct: True
